# SIC embedding regression laboratory

This replaces the historical work-in-progress embedding notebook. Start with
[the tutorial](Tutorial.ipynb) for installation and conventions. Every circuit
below compares *all* physical outcomes to direct Born probabilities, and checks
that padding stays empty. No postselection is used.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import cirq
from pathlib import Path
import tempfile

from cirq_sic.measurements import Measurement, sic_povm, wh_measurement
from cirq_sic.workflow import (
    Experiment, ExperimentResult, tomography_experiment, entropy_experiment,
    wh_entropy_experiment, sky_ground_experiment, triple_product_experiment,
    compile_circuit,
)
from cirq_sic.analysis import stabilizer_renyi_entropy, born_rule_diagnostics
from cirq_sic.triples import triple_products, triple_product_state_space, sic_state_constraints

rng = np.random.default_rng(2026)
plt.rcParams.update({'figure.figsize': (7, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print('Cirq', cirq.__version__)

Cirq 1.3.0


## 1. A qutrit SIC on qubit hardware

A qutrit ($d=3$) needs two system qubits. Its nine-outcome SIC needs four measured
qubits in the dense Naimark construction. Output bit strings 9–15 are padding and
have zero ideal probability. They are **retained as leakage** if a noisy device
produces them.

The frame columns $f_k$ obey $\sum_k f_k f_k^\dagger=I$. Therefore
$V_{kj}=\overline{f_{jk}}$ is an isometry. Completing its columns to a unitary gives
an ordinary qubit measurement with the desired probabilities. Input columns must
be placed at $j2^{n_\mathrm{ancilla}}$, because the zero ancillas come *after* the
system. Confusing these columns, or measuring a larger-dimension WH orbit after
padding the fiducial, changes the POVM.

`Measurement` accepts any complete rank-one frame, `wh_measurement` accepts a
normalized fiducial, and `sic_povm` additionally verifies the SIC overlaps.

In [2]:
rows = []
for d in [2, 3, 4, 5, 6, 7, 8]:
    measurement = sic_povm(d)
    psi = rng.normal(size=d) + 1j*rng.normal(size=d)
    psi /= np.linalg.norm(psi)
    plan = tomography_experiment(psi, measurement)
    data = plan.simulate()['measurement']
    error = np.max(abs(data.valid - measurement.probabilities(psi)))
    rows.append((d, measurement.num_system_qubits, measurement.num_qubits,
                 measurement.num_outcomes, len(data.probabilities), error, data.leakage))
    assert error < 1e-10 and data.leakage < 1e-12
print('d  system  measured  outcomes  bitstrings  max error  leakage')
for row in rows:
    print('%d  %d  %d  %d  %d  %.2e  %.2e' % row)

d  system  measured  outcomes  bitstrings  max error  leakage
2  1  2  4  4  0.00e+00  0.00e+00
3  2  4  9  16  5.55e-17  0.00e+00
4  2  4  16  16  8.33e-17  0.00e+00
5  3  5  25  32  2.78e-17  0.00e+00
6  3  6  36  64  2.78e-17  0.00e+00
7  3  6  49  64  5.55e-17  0.00e+00
8  3  6  64  64  2.08e-17  0.00e+00


## Mixed input and arbitrary rank-one frames

The embedding works for any complete weighted frame, including a non-SIC WH
orbit. Tomography additionally needs informational completeness. Density matrices
are purified with separate environment qubits.

In [3]:
d = 3
ket = np.array([1, 1j, 2], dtype=complex) / np.sqrt(6)
rho = .6*np.outer(ket, ket.conj()) + .4*np.eye(d)/d
measurement = wh_measurement(ket)
plan = tomography_experiment(rho, measurement)
data = plan.simulate()['measurement']
assert np.allclose(data.valid, measurement.probabilities(rho), atol=1e-10)
assert np.allclose(plan.analyze(plan.simulate())['density_matrix'], rho, atol=1e-9)
print('Mixed-state qubits:', len(plan.circuits[0].qubit_order))

Mixed-state qubits: 6


## Leakage is data

For a qutrit SIC the physical probabilities occupy outcomes 0–8; 9–15 remain in
`OutcomeData.probabilities` and `counts`. `Experiment.analyze` rejects appreciable
leakage because silently conditioning on successful outputs changes the model.
Inspect leakage per circuit when stress testing a device.

## 5. Compile, inspect, and iterate

The mathematical construction contains dense unitary gates. `compile_circuit`
synthesizes one- and two-qubit operations and targets CZ gates by default. The
synthesis uses a stable Schur-based decomposition for degenerate padded blocks.
This is a correctness reference, not a low-depth SIC synthesis algorithm.
A SIC dilation stores $O(d^4)$ complex numbers and dense completion costs roughly
$O(d^6)$; hardware gate counts grow quickly. Arbitrary dimension means the
encoding has no power-of-two restriction, not that every dimension is affordable.

Gate synthesis does not route or validate against a particular device topology.
After your own placement/routing, supply a `cirq.Sampler` to `plan.run(...)`.
If routing changes qubits, construct updated `CircuitSpec` layouts explicitly.
Do not reuse the old output order. No cloud submission is performed here.

In [4]:
plan = tomography_experiment(np.eye(3)[0])
compiled = plan.transform(compile_circuit)
assert np.allclose(plan.simulate()['measurement'].probabilities,
                   compiled.simulate()['measurement'].probabilities, atol=1e-7)
print('Synthesized operations:', len(list(compiled.circuits[0].circuit.all_operations())))

Synthesized operations: 366
